In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install transformers==5.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 95.0 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
%pip show transformers

Name: transformers
Version: 5.3.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: peft, sentence-transformers


In [ ]:
import torch
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

intent_datasets = load_dataset(
    "json",
    data_files={
        "train": "/content/drive/MyDrive/Colab Notebooks/SHR_data/processed/convs_intents+NER_train.jsonl",
        "valid": "/content/drive/MyDrive/Colab Notebooks/SHR_data/processed/convs_intents+NER_valid.jsonl",
        "test": "/content/drive/MyDrive/Colab Notebooks/SHR_data/processed/convs_intents+NER_test.jsonl",
    },
)

le_intents = LabelEncoder()
all_intents = intent_datasets["train"]["intent"]
le_intents.fit(all_intents)
intent_datasets = intent_datasets.map(
    lambda x: {"labels": le_intents.transform(x["intent"])},
    batched=True,
)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize(sample):
    return tokenizer(
        sample["utterance"],
        truncation=True,
        max_length=64,
    )


intent_datasets = intent_datasets.map(
    tokenize,
    batched=True,
    remove_columns=["turn_id", "speaker", "utterance", "intent", "ner_tags"],
)
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=len(le_intents.classes_)
).to(device)

training_args = TrainingArguments(
    output_dir="./result",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    logging_dir="./logs",
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1_score, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1-score": f1_score,
    }


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=intent_datasets["train"],
    eval_dataset=intent_datasets["valid"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

trainer.train()

result = trainer.evaluate(eval_dataset=intent_datasets["test"])
df_result = pd.DataFrame(result)
output_paths = {
    "result": "/content/drive/MyDrive/Colab Notebooks/results/metrics_performance.csv",
    "bert": "/content/drive/MyDrive/Colab Notebooks/models/intent_model",
}
df_result.to_csv(output_paths["result"], index=False)
model.save_pretrained(output_paths["bert"])
tokenizer.save_pretrained(output_paths["bert"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

Epoch,Training Loss,Validation Loss
